# DeepSeek V4 Pro: 256K disaggregated versus aggregated

A matched live test on the same 16 H200 GPUs, checkpoint, image and PVCs.
Each topology serves the same eight three-turn sessions at concurrency 4: **24 requests
per topology, 256,000–256,074 prompt tokens, up to 256 output tokens**.
The context window is 262,144 tokens. Non-thinking mode, temperature 0.

The CPU client runs inside the cluster and uses the same frontend ClusterIP.
Both worker KV caches are cleared before each run; follow-up turns can reuse
context. Model/kernel warmup pilots are excluded. Disaggregated runs first;
aggregated runs second. One run per topology fits the approximately 30-minute
measurement budget: these results do not establish repeatability or reliable p99s.

The final cluster is left **aggregated**. Raw measurements, metadata and worker
Prometheus snapshots are stored in `tracks/nvidia-dynamo/studies/deepseek-v4-pro-256k-comparison/`.
Run all cells using the repository `.venv` kernel. No load is generated here.

In [ ]:
from pathlib import Path
import os, json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'benchmarks').is_dir())
sys.path.insert(0, str(ROOT))
from benchmarks.collect import collect
RESULTS = Path(os.environ.get('DEEPSEEK_256K_RESULTS', ROOT / 'tracks/nvidia-dynamo/studies/deepseek-v4-pro-256k-comparison'))
EXPORT = RESULTS / 'analysis'
EXPORT.mkdir(parents=True, exist_ok=True)
summary_path = collect(RESULTS)
runs = pd.read_csv(summary_path) if summary_path else pd.DataFrame()
LABELS = {'dynamo-disagg-k8s': 'Disaggregated (1P + 1D)', 'dynamo-agg-k8s': 'Aggregated (2 replicas)'}
ORDER = list(LABELS.values())
plt.style.use('seaborn-v0_8-whitegrid')
print(f'{len(runs)} runs from {RESULTS}')


## Audit the experiment

Failures and invalid measurements must be visible before inspecting speed. Confirm
that the dataset, budgets, GPU count and sampling options match. Deployment hashes
must differ because the worker topology differs. The full deployment snapshots
are in each run's `metadata.json`.

In [ ]:
requests = pd.DataFrame()
if not runs.empty:
    display(runs[['run_id','technology','requests','failed_requests','invalid_measurements',
                  'duration_s','concurrency','prompt_tokens_mean','completion_tokens_mean',
                  'dataset_sha256','cache_state','gpu_count']])
    for field in ['dataset_sha256','model','model_revision','image','gpu_count','concurrency',
                  'sessions','max_model_len','requested_output_tokens','min_input_tokens',
                  'extra_body_json','temperature','cache_state','session_rate','think_time_s']:
        assert runs[field].nunique(dropna=False) == 1, f'Mismatched setting: {field}'
    frames=[]
    for row in runs.itertuples():
        frame=pd.read_csv(RESULTS / row.run_id / 'requests.csv')
        frame['mode']=LABELS[row.technology]
        frames.append(frame)
    requests=pd.concat(frames,ignore_index=True)
    display(requests.groupby('mode').agg(requests=('success','size'),
        successful=('success','sum'), valid=('measurement_valid','sum'),
        min_prompt=('prompt_tokens','min'), max_prompt=('prompt_tokens','max'),
        mean_output=('completion_tokens','mean')))
    if set(runs.technology) == set(LABELS):
        a=requests[requests['mode']==ORDER[0]].set_index(['session_id','turn'])
        b=requests[requests['mode']==ORDER[1]].set_index(['session_id','turn'])
        assert a.index.is_unique and b.index.is_unique
        assert set(a.index)==set(b.index), 'Different sessions/turns'
        assert a.request_sha256.sort_index().equals(b.request_sha256.sort_index()), 'Request bodies differ'
        print('Verified identical request bodies for every paired session/turn.')
    else:
        print('Comparison incomplete: both topology runs are required.')


## Whole-run throughput and latency

Output throughput uses actual generated output tokens divided by wall time. The
256-token budget is a ceiling; early stopping and small numerical differences can
produce different output lengths. Report request throughput and output lengths
alongside token throughput. SGLang does not expose exact token-resolved ITL here;
TPOT is the average inter-token time within each response.

In [ ]:
comparison=pd.DataFrame()
if not runs.empty:
    comparison=runs[(runs.failed_requests==0)&(runs.invalid_measurements==0)].copy()
    comparison['mode']=comparison.technology.map(LABELS)
    comparison['output_tps_per_gpu']=comparison.output_throughput_tps/comparison.gpu_count
    comparison['ttft_p50_s']=comparison.ttft_ms_p50/1000
    comparison['ttft_p95_s']=comparison.ttft_ms_p95/1000
    display(comparison[['mode','duration_s','requests','request_throughput_rps',
        'output_throughput_tps','output_tps_per_gpu','ttft_p50_s','ttft_p95_s',
        'tpot_ms_p50','tpot_ms_p95','completion_tokens_mean']])
    comparison.to_csv(EXPORT/'topology-comparison.csv',index=False)
    if not comparison.empty:
        fig,axes=plt.subplots(1,3,figsize=(16,4),layout='constrained')
        for ax,col,title in zip(axes,['output_throughput_tps','ttft_p50_s','tpot_ms_p50'],
            ['Output tokens/s ↑','Median TTFT (s) ↓','Median TPOT (ms) ↓']):
            bars=ax.bar(comparison['mode'],comparison[col],color=['#217c91','#d47c36'][:len(comparison)])
            ax.bar_label(bars,fmt='%.2f',padding=3); ax.margins(y=.12)
            ax.set_title(title); ax.tick_params(axis='x',labelrotation=12)
        fig.savefig(EXPORT/'topology-comparison.png',dpi=180,bbox_inches='tight')
        plt.show()


## First turns versus context reuse

First turns begin after a cache reset and have unique session prefixes; later turns
can reuse their session context. Actual cache hits and eviction still depend on
the topology and scheduler. Separating these phases avoids hiding long-prefill
latency behind fast follow-up requests. TTFT includes queueing inside the serving
stack; client concurrency-slot wait is recorded separately.

In [ ]:
if not requests.empty:
    good=requests[(requests.success==True)&(requests.measurement_valid==True)].copy()
    good['phase']=np.where(good.turn==0,'First turn','Follow-up')
    good['ttft_s']=good.ttft_ms/1000
    good['e2e_s']=good.e2e_ms/1000
    phase=good.groupby(['mode','phase']).agg(
        requests=('turn','size'), ttft_p50_s=('ttft_s','median'),
        ttft_p95_s=('ttft_s',lambda x:x.quantile(.95)),
        e2e_p50_s=('e2e_s','median'), tpot_p50_ms=('tpot_ms','median'),
        mean_cached_tokens=('cached_tokens','mean'), mean_output_tokens=('completion_tokens','mean')).reset_index()
    display(phase)
    phase.to_csv(EXPORT/'latency-by-phase.csv',index=False)
    fig,axes=plt.subplots(1,2,figsize=(13,5),layout='constrained')
    for ax,phase_name in zip(axes,['First turn','Follow-up']):
        for mode in ORDER:
            values=np.sort(good.loc[(good['mode']==mode)&(good.phase==phase_name),'ttft_s'].dropna())
            if len(values):ax.step(values,np.arange(1,len(values)+1)/len(values),where='post',label=mode)
        ax.set_title(phase_name+' TTFT');ax.set_xlabel('Seconds');ax.set_ylabel('Fraction of requests');ax.legend(fontsize=8)
    axes[1].set_xscale('log'); axes[1].set_xlabel('Seconds (log scale)')
    fig.savefig(EXPORT/'ttft-by-phase.png',dpi=180,bbox_inches='tight')
    plt.show()
    good.to_csv(EXPORT/'valid-requests.csv',index=False)


## Interpretation and limits

- This compares **one TP8 prefill + one TP8 decode** against **two TP8 aggregated
  replicas**, with all 16 GPUs allocated in either case. Each worker admits four
  requests; the client caps total simultaneous sessions at four.
- Synthetic operations/policy conversations are replayed; answers and tool quality
  are not scored. Output lengths are not forced to be identical.
- Cache-reset acknowledgements, image/model manifests, raw streaming events and
  metrics snapshots accompany the results. Cached input throughput is not fresh
  prefill throughput.
- One run per topology and fixed topology order leave run-to-run variation and
  temporal effects unresolved. This is a measured workload comparison, not a
  general claim that either topology is faster for every workload.
- The 256K pilots are stored separately and excluded from the charts.

Exports are saved in the result folder's `analysis/` directory.